# Medical Insurance Cost Prediction

**Target:** `annual_medical_cost`

This notebook:
- Loads and explores the medical insurance dataset
- Removes ID, duplicate/derived and leakage-prone columns
- Handles categorical features with OneHotEncoder
- Trains multiple regression algorithms
- Compares MAE, RMSE and R²
- Selects the best model based on R²


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
    RandomForestRegressor,
    StackingRegressor,
    VotingRegressor
)
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("Libraries imported successfully")


In [ ]:
# Load dataset
df = pd.read_csv("medical_insurance.csv")

print("Shape:", df.shape)
display(df.head())


In [ ]:
# Dataset information
display(df.info())
print("\nMissing values:")
display(df.isnull().sum().sort_values(ascending=False).head(15))


In [ ]:
# Target distribution
plt.figure(figsize=(8, 5))
plt.hist(df["annual_medical_cost"], bins=50)
plt.xlabel("Annual Medical Cost")
plt.ylabel("Frequency")
plt.title("Distribution of Annual Medical Cost")
plt.show()

display(df["annual_medical_cost"].describe())


## Feature Selection

The following columns are removed:

- `person_id`: identifier
- `annual_premium`, `monthly_premium`: closely related insurance-price variables
- `is_high_risk`, `chronic_count`: derived variables
- `claims_count`, `avg_claim_amount`, `total_claims_paid`: claims/outcome-related variables that can introduce leakage when predicting future medical cost


In [ ]:
target = "annual_medical_cost"

drop_cols = [
    "person_id",
    "annual_premium",
    "monthly_premium",
    "is_high_risk",
    "chronic_count",
    "claims_count",
    "avg_claim_amount",
    "total_claims_paid"
]

# Keep target separate
X = df.drop(columns=drop_cols + [target])
y = df[target]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nRemaining columns:")
print(X.columns.tolist())


In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training data:", X_train.shape)
print("Testing data :", X_test.shape)


In [ ]:
# Identify numerical and categorical columns
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print("Numerical features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("\nCategorical columns:")
print(categorical_features)


In [ ]:
# Preprocessing
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

print("Preprocessor ready")


# 1. Linear Regression

A basic linear regression model used as the baseline.


In [ ]:
linear_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

linear_model.fit(X_train, y_train)
pred_linear = linear_model.predict(X_test)

print("Linear Regression")
print("MAE :", mean_absolute_error(y_test, pred_linear))
print("RMSE:", np.sqrt(mean_squared_error(y_test, pred_linear)))
print("R²  :", r2_score(y_test, pred_linear))


# 2. Ridge Regression

Ridge adds L2 regularization and is useful when features are correlated.


In [ ]:
ridge_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", Ridge(alpha=10.0))
])

ridge_model.fit(X_train, y_train)
pred_ridge = ridge_model.predict(X_test)

print("Ridge Regression")
print("MAE :", mean_absolute_error(y_test, pred_ridge))
print("RMSE:", np.sqrt(mean_squared_error(y_test, pred_ridge)))
print("R²  :", r2_score(y_test, pred_ridge))


# 3. Lasso Regression

Lasso uses L1 regularization and can shrink less useful coefficients toward zero.


In [ ]:
lasso_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", Lasso(alpha=10.0, max_iter=10000))
])

lasso_model.fit(X_train, y_train)
pred_lasso = lasso_model.predict(X_test)

print("Lasso Regression")
print("MAE :", mean_absolute_error(y_test, pred_lasso))
print("RMSE:", np.sqrt(mean_squared_error(y_test, pred_lasso)))
print("R²  :", r2_score(y_test, pred_lasso))


# 4. Decision Tree Regressor


In [ ]:
dt_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DecisionTreeRegressor(
        max_depth=20,
        min_samples_split=10,
        random_state=42
    ))
])

dt_model.fit(X_train, y_train)
pred_dt = dt_model.predict(X_test)

print("Decision Tree Regressor")
print("MAE :", mean_absolute_error(y_test, pred_dt))
print("RMSE:", np.sqrt(mean_squared_error(y_test, pred_dt)))
print("R²  :", r2_score(y_test, pred_dt))


# 5. Random Forest Regressor

An ensemble of decision trees that usually performs well on nonlinear tabular data.


In [ ]:
rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=200,
        max_depth=None,
        min_samples_split=5,
        n_jobs=-1,
        random_state=42
    ))
])

rf_model.fit(X_train, y_train)
pred_rf = rf_model.predict(X_test)

print("Random Forest Regressor")
print("MAE :", mean_absolute_error(y_test, pred_rf))
print("RMSE:", np.sqrt(mean_squared_error(y_test, pred_rf)))
print("R²  :", r2_score(y_test, pred_rf))


# 6. Extra Trees Regressor


In [ ]:
extra_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", ExtraTreesRegressor(
        n_estimators=200,
        min_samples_split=5,
        n_jobs=-1,
        random_state=42
    ))
])

extra_model.fit(X_train, y_train)
pred_extra = extra_model.predict(X_test)

print("Extra Trees Regressor")
print("MAE :", mean_absolute_error(y_test, pred_extra))
print("RMSE:", np.sqrt(mean_squared_error(y_test, pred_extra)))
print("R²  :", r2_score(y_test, pred_extra))


# 7. Gradient Boosting Regressor


In [ ]:
gb_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=4,
        random_state=42
    ))
])

gb_model.fit(X_train, y_train)
pred_gb = gb_model.predict(X_test)

print("Gradient Boosting Regressor")
print("MAE :", mean_absolute_error(y_test, pred_gb))
print("RMSE:", np.sqrt(mean_squared_error(y_test, pred_gb)))
print("R²  :", r2_score(y_test, pred_gb))


# 8. HistGradientBoosting Regressor

A fast gradient boosting implementation available directly in scikit-learn.


In [ ]:
# HistGradientBoosting works best with dense data.
# This separate preprocessor produces dense encoded features.
dense_preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ]), categorical_features)
])

hist_model = Pipeline([
    ("preprocessor", dense_preprocessor),
    ("model", HistGradientBoostingRegressor(
        max_iter=200,
        learning_rate=0.05,
        max_leaf_nodes=31,
        random_state=42
    ))
])

hist_model.fit(X_train, y_train)
pred_hist = hist_model.predict(X_test)

print("HistGradientBoosting Regressor")
print("MAE :", mean_absolute_error(y_test, pred_hist))
print("RMSE:", np.sqrt(mean_squared_error(y_test, pred_hist)))
print("R²  :", r2_score(y_test, pred_hist))


# 9. KNN Regressor

KNN uses nearby observations to predict the target. Scaling is important, which is already handled in preprocessing.


In [ ]:
knn_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", KNeighborsRegressor(
        n_neighbors=10,
        weights="distance",
        n_jobs=-1
    ))
])

knn_model.fit(X_train, y_train)
pred_knn = knn_model.predict(X_test)

print("KNN Regressor")
print("MAE :", mean_absolute_error(y_test, pred_knn))
print("RMSE:", np.sqrt(mean_squared_error(y_test, pred_knn)))
print("R²  :", r2_score(y_test, pred_knn))


# 10. Support Vector Regression

SVR can be computationally expensive on 100,000 rows. Therefore, this section trains on a sample of the training data.


In [ ]:
# Use a sample for SVR because full 100k-row SVR can be extremely slow
svr_sample_size = min(15000, len(X_train))
X_svr = X_train.sample(svr_sample_size, random_state=42)
y_svr = y_train.loc[X_svr.index]

svr_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", SVR(
        kernel="rbf",
        C=100,
        epsilon=0.1
    ))
])

svr_model.fit(X_svr, y_svr)
pred_svr = svr_model.predict(X_test)

print(f"SVR trained on {svr_sample_size:,} rows")
print("MAE :", mean_absolute_error(y_test, pred_svr))
print("RMSE:", np.sqrt(mean_squared_error(y_test, pred_svr)))
print("R²  :", r2_score(y_test, pred_svr))


# 11. XGBoost Regressor

XGBoost is optional. If it is not installed, run:

`pip install xgboost`


In [ ]:
try:
    from xgboost import XGBRegressor

    xgb_model = Pipeline([
        ("preprocessor", preprocessor),
        ("model", XGBRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="reg:squarederror",
            n_jobs=-1,
            random_state=42
        ))
    ])

    xgb_model.fit(X_train, y_train)
    pred_xgb = xgb_model.predict(X_test)

    print("XGBoost Regressor")
    print("MAE :", mean_absolute_error(y_test, pred_xgb))
    print("RMSE:", np.sqrt(mean_squared_error(y_test, pred_xgb)))
    print("R²  :", r2_score(y_test, pred_xgb))

except ImportError:
    print("XGBoost is not installed. Run: pip install xgboost")
    pred_xgb = None


# Model Comparison


In [ ]:
predictions = {
    "Linear Regression": pred_linear,
    "Ridge Regression": pred_ridge,
    "Lasso Regression": pred_lasso,
    "Decision Tree": pred_dt,
    "Random Forest": pred_rf,
    "Extra Trees": pred_extra,
    "Gradient Boosting": pred_gb,
    "HistGradientBoosting": pred_hist,
    "KNN": pred_knn,
    "SVR": pred_svr
}

if pred_xgb is not None:
    predictions["XGBoost"] = pred_xgb

results = []

for name, pred in predictions.items():
    results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
        "R2 Score": r2_score(y_test, pred)
    })

results_df = pd.DataFrame(results).sort_values(
    by="R2 Score",
    ascending=False
).reset_index(drop=True)

display(results_df)


In [ ]:
# Visual comparison of R²
plt.figure(figsize=(11, 6))
plt.barh(results_df["Model"], results_df["R2 Score"])
plt.xlabel("R² Score")
plt.ylabel("Model")
plt.title("Model Comparison - R² Score")
plt.gca().invert_yaxis()
plt.show()


In [ ]:
# Best model
best_model_name = results_df.loc[0, "Model"]
best_r2 = results_df.loc[0, "R2 Score"]
best_mae = results_df.loc[0, "MAE"]
best_rmse = results_df.loc[0, "RMSE"]

print("Best Model:", best_model_name)
print("R² Score:", best_r2)
print("MAE:", best_mae)
print("RMSE:", best_rmse)


## Actual vs Predicted for Best Model


In [ ]:
best_predictions = predictions[best_model_name]

plt.figure(figsize=(8, 6))
plt.scatter(y_test, best_predictions, alpha=0.25)
plt.xlabel("Actual Annual Medical Cost")
plt.ylabel("Predicted Annual Medical Cost")
plt.title(f"Actual vs Predicted - {best_model_name}")
plt.show()


## Conclusion

The model with the highest **R² score** is selected as the best model.

For a medical-cost prediction project, avoid using variables that are only known **after** the medical-cost outcome occurs. Otherwise, the model can suffer from data leakage and produce unrealistically high evaluation scores.
